# Notebook 03 — BiLSTM Training

Interactive walkthrough of the BiLSTM shot predictor training.

**What we cover:**
1. Load and inspect the sequence data
2. Build the BiLSTM model and inspect architecture
3. Train with cross-validation
4. Plot training curves
5. Visualize attention weights (which frames matter most?)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.modeling.bilstm_model import ShotPredictor, build_model, count_parameters

sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded | Device: {device}')

## 1. Load Sequence Data

In [ ]:
seq_dir = Path('../data/processed/sequences')

X_train = np.load(seq_dir / 'X_train.npy')
X_val   = np.load(seq_dir / 'X_val.npy')
X_test  = np.load(seq_dir / 'X_test.npy')
y_train = np.load(seq_dir / 'y_train.npy')
y_val   = np.load(seq_dir / 'y_val.npy')
y_test  = np.load(seq_dir / 'y_test.npy')

feat_names = (seq_dir / 'feature_names.txt').read_text().strip().split('\n')

print(f'X_train shape: {X_train.shape}  (shots × timesteps × features)')
print(f'X_val   shape: {X_val.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'Features ({len(feat_names)}): {feat_names}')
print(f'\nClass balance:')
for split, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'  {split:6s}: makes={int(y.sum()):3d}  misses={int((1-y).sum()):3d}  make%={y.mean():.1%}')

## 2. Sequence Visualization

Visualizing a few sequences: how do biomechanical angles change over time for makes vs misses?

In [ ]:
# Show elbow angle over time for makes vs misses
feat_idx = feat_names.index('elbow_angle') if 'elbow_angle' in feat_names else 0
feat_name = feat_names[feat_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
T = X_train.shape[1]
x_axis = np.arange(T)

for ax, outcome, label, color in [(axes[0], 1, 'Make', '#2ecc71'), 
                                   (axes[1], 0, 'Miss', '#e74c3c')]:
    mask = y_train == outcome
    samples = X_train[mask][:20, :, feat_idx]  # up to 20 examples
    for seq in samples:
        ax.plot(x_axis, seq, color=color, alpha=0.2, linewidth=1)
    if len(samples) > 0:
        ax.plot(x_axis, samples.mean(0), color=color, linewidth=2.5, 
               label=f'{label} mean (n={len(samples)})')
    ax.axvline(T-1, color='gray', linestyle=':', alpha=0.7, label='Release frame')
    ax.set_title(f'{feat_name.replace("_"," ").title()} — {label}', fontweight='bold')
    ax.set_xlabel('Frame (0 = shot start, T-1 = release)')
    ax.set_ylabel('Normalized value')
    ax.legend()

plt.suptitle(f'Sequence Visualization: {feat_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
n_features = X_train.shape[2]
model = build_model(input_size=n_features, hidden_size=128, num_layers=2, dropout=0.3)
model = model.to(device)

print(model)
print(f'\nTotal trainable parameters: {count_parameters(model):,}')

## 4. Quick Training Run (5 epochs for demo)

For full training, use: `python src/modeling/train_bilstm.py --epochs 100`

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

X_tr = torch.FloatTensor(X_train).to(device)
y_tr = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_vl = torch.FloatTensor(X_val).to(device)
y_vl = torch.FloatTensor(y_val)

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = torch.nn.BCELoss()

history = {'train_loss': [], 'val_auc': []}
DEMO_EPOCHS = 5

for epoch in range(1, DEMO_EPOCHS + 1):
    model.train()
    total_loss = 0
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        probs = model(X_vl.to(device)).cpu().squeeze().numpy()
    try:
        auc = roc_auc_score(y_vl.numpy(), probs)
    except Exception:
        auc = 0.5
    
    history['train_loss'].append(total_loss / len(loader))
    history['val_auc'].append(auc)
    print(f'Epoch {epoch}/{DEMO_EPOCHS} | loss={total_loss/len(loader):.4f} | val_auc={auc:.4f}')

print('\n(For full training: python src/modeling/train_bilstm.py --epochs 100)')

## 5. Attention Weight Visualization

Which frames in the shot sequence does the model focus on?

In [ ]:
model.eval()
n_show = min(6, len(X_train))
sample_idx = np.random.choice(len(X_train), n_show, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

with torch.no_grad():
    for i, (ax, idx) in enumerate(zip(axes, sample_idx)):
        x_sample = torch.FloatTensor(X_train[idx]).unsqueeze(0).to(device)
        true_label = int(y_train[idx])
        
        prob, attn_weights = model(x_sample, return_attention=True)
        prob_val = float(prob.item())
        attn = attn_weights.cpu().squeeze().numpy()
        
        # Bar chart of attention weights
        color = '#2ecc71' if true_label == 1 else '#e74c3c'
        ax.bar(range(len(attn)), attn, color=color, alpha=0.8)
        ax.set_title(
            f'True: {"Make" if true_label else "Miss"} | '
            f'Pred prob: {prob_val:.2f}',
            fontsize=9, fontweight='bold'
        )
        ax.set_xlabel('Frame')
        ax.set_ylabel('Attention weight')
        ax.axvline(len(attn)-1, color='gray', linestyle=':', alpha=0.7)

plt.suptitle('Attention Weights per Frame (gray line = release)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Interpretation: Higher bars = frames the model weights more heavily for prediction')